In [1]:
import pandas as pd
import re
import datetime
import logging
import pathlib
import global3d_track as g3d

import matplotlib.pyplot as plt

datadir = pathlib.Path(f'/work/bb1153/b382635/data/track_statistics/dcc_frozen_1e4/tropics')
outdir = pathlib.Path(f'/work/bb1153/b382635/plots/model_paper/grl_submission/data/statistics_L1')
outdir.mkdir(parents=1, exist_ok=1)


### load and aggregate key results

#### PROCESS

In [2]:
import statistics

INFO:root:Region to regrid: (-180, 180, -15, 15)


In [3]:
files = list(datadir.glob('*'))
N = len(files)
# print(N, 'DCCs ready for the making')

In [4]:
# test file names
regex = r'_(\d+)\.0.nc$'
isolated_dccs = pd.read_csv('/work/bb1153/b382635/data/final_tracks/dcc_frozen_1e4/tropics/track_stats/dcc_validity.csv', index_col=0).valid
isolated_dccs = isolated_dccs.index[isolated_dccs]
gfiles = []
misc = []
for f in files:
    try:
        dcc = int(re.search(regex, pathlib.Path(f).name).group(1))
        if dcc in isolated_dccs:
            gfiles.append(f)
    except:
        misc.append(f)
print(len(gfiles), 'valid isolated DCCs')

4519 valid isolated DCCs


In [5]:
SIZE = 50

for i in range(1,int(N/SIZE)+2):
    # start batch
    logging.info('batch %s' %i)
    t0 = datetime.datetime.now()
    batch = i
    results_path = outdir / f'b{batch}s{SIZE}.nc'
    data_params = dict(files=gfiles, batch=batch, size=SIZE)

    # exists
    if results_path.exists():
        continue

    # calculate
    stats = g3d.analysis.helpers.load_stats(apply=statistics.calculate, **data_params)
    stats["system"] = stats["system"].assign_attrs(long_name="unique identifier for a tracked deep convective cloud")
    stats["core"] = stats["core"].assign_attrs(long_name="unique identifier for a convective core associated with a tracked deep convective cloud")

    logging.info('calculated')

    # save
    stats.to_netcdf(results_path)
    tdelta = datetime.datetime.now() - t0
    logging.info('took %s, saved to %s' %(tdelta, results_path))
    stats.close()



INFO:root:batch 1
INFO:root:calculated
INFO:root:took 0:00:06.645538, saved to /work/bb1153/b382635/plots/model_paper/grl_submission/data/statistics_L1/b1s50.nc
INFO:root:batch 2
INFO:root:calculated
INFO:root:took 0:00:06.253917, saved to /work/bb1153/b382635/plots/model_paper/grl_submission/data/statistics_L1/b2s50.nc
INFO:root:batch 3
INFO:root:calculated
INFO:root:took 0:00:10.616664, saved to /work/bb1153/b382635/plots/model_paper/grl_submission/data/statistics_L1/b3s50.nc
INFO:root:batch 4
INFO:root:calculated
INFO:root:took 0:00:10.985780, saved to /work/bb1153/b382635/plots/model_paper/grl_submission/data/statistics_L1/b4s50.nc
INFO:root:batch 5
INFO:root:calculated
INFO:root:took 0:00:10.101575, saved to /work/bb1153/b382635/plots/model_paper/grl_submission/data/statistics_L1/b5s50.nc
INFO:root:batch 6
INFO:root:calculated
INFO:root:took 0:00:10.752197, saved to /work/bb1153/b382635/plots/model_paper/grl_submission/data/statistics_L1/b6s50.nc
INFO:root:batch 7
INFO:root:calcul

ValueError: no datasets loaded successfully